In [1]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
from datetime import datetime
import urllib.parse

Asked google gemini to scrape personal finance boggle heads discussions during Nov 2021 and July 2022. Was able to get stackexchange discussions pretty straightforward manner. For bogleheads, we are trying to scrape it in a straightforward manner 

In [3]:
import os

# Get the current working directory
cwd = os.getcwd()
print("Current Working Directory:", cwd)

Current Working Directory: c:\Users\k639s258\OneDrive - University of Kansas\RootFolder\Research\Jinhang_\EUI


In [4]:
# Configuration
BASE_URL = "https://www.bogleheads.org/forum/viewforum.php?f=2"
DOMAIN = "https://www.bogleheads.org/forum/"
START_DATE = datetime(2021, 11, 23)
END_DATE = datetime(2022, 6, 25)

headers = {
    'User-Agent': 'KarthikS (Contact: yetanotherdatascientist@gmail.com)'
}

def get_post_data(thread_url):
    """Scrapes individual posts within a thread that fall in the date range."""
    posts_data = []
    response = requests.get(thread_url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Each post is contained in a div with class 'postbody'
    posts = soup.find_all('div', class_='postbody')
    
    for post in posts:
        # Extract timestamp
        time_tag = post.find('time')
        if time_tag:
            post_date = datetime.fromisoformat(time_tag['datetime'].split('+')[0])
            
            # Check if post is within our target window
            if START_DATE <= post_date <= END_DATE:
                author = post.find('p', class_='author').find('strong').text if post.find('strong') else "Unknown"
                content = post.find('div', class_='content').text.strip()
                
                posts_data.append({
                    'date': post_date,
                    'author': author,
                    'text': content,
                    'url': thread_url
                })
    return posts_data

def scrape_bogleheads_range(max_pages=10):
    all_results = []
    
    for i in range(max_pages):
        start_val = i * 50
        url = f"{BASE_URL}&start={start_val}"
        print(f"Checking forum page {i+1}...")
        
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        threads = soup.find_all('li', class_='row')
        
        for thread in threads:
            # Check the date of the LAST post in the thread to see if it's worth entering
            last_post_time_tag = thread.find('time')
            if last_post_time_tag:
                last_active = datetime.fromisoformat(last_post_time_tag['datetime'].split('+')[0])
                
                # If the thread was active during or after our start date, we check it
                if last_active >= START_DATE:
                    thread_link = thread.find('a', class_='topictitle')['href'].replace('./', DOMAIN)
                    print(f"  Processing thread: {thread_link[:60]}...")
                    
                    # Scrape the posts inside
                    thread_posts = get_post_data(thread_link)
                    all_results.extend(thread_posts)
                    
                    # Avoid overwhelming the server
                    time.sleep(1.5)
                
                # If we encounter threads that haven't been active since before our window, 
                # we can likely stop (since threads are sorted by last activity).
                if last_active < START_DATE:
                    print("Reached threads older than target range. Stopping.")
                    return pd.DataFrame(all_results)

    return pd.DataFrame(all_results)



Above did not work, so trying again

In [5]:
# Execute
df = scrape_bogleheads_range(max_pages=100)


Checking forum page 1...
Checking forum page 2...
Checking forum page 3...
Checking forum page 4...
Checking forum page 5...
Checking forum page 6...
Checking forum page 7...
Checking forum page 8...
Checking forum page 9...
Checking forum page 10...
Checking forum page 11...
Checking forum page 12...
Checking forum page 13...
Checking forum page 14...
Checking forum page 15...
Checking forum page 16...
Checking forum page 17...
Checking forum page 18...
Checking forum page 19...
Checking forum page 20...
Checking forum page 21...
Checking forum page 22...
Checking forum page 23...
Checking forum page 24...
Checking forum page 25...
Checking forum page 26...
Checking forum page 27...
Checking forum page 28...
Checking forum page 29...
Checking forum page 30...
Checking forum page 31...
Checking forum page 32...
Checking forum page 33...
Checking forum page 34...
Checking forum page 35...
Checking forum page 36...
Checking forum page 37...
Checking forum page 38...
Checking forum page 3

In [14]:
# Configuration
FORUM_URL = "https://www.bogleheads.org/forum/viewforum.php?f=2&start=27000"
BASE_URL = "https://www.bogleheads.org/forum/"
START_DATE = datetime(2021, 11, 23)
END_DATE = datetime(2022, 6, 25)

# Use a session to persist cookies and headers
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
})

def parse_date(date_str):
    """Attempts to parse various date formats found on the forum."""
    try:
        # Most common ISO-like format in 'datetime' attribute
        return datetime.fromisoformat(date_str.split('+')[0])
    except:
        return None

def get_thread_posts(thread_url):
    """Scrapes all posts within a specific thread."""
    posts_data = []
    response = session.get(thread_url)
    if response.status_code != 200:
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    posts = soup.find_all('div', class_='post')
    
    for post in posts:
        time_tag = post.find('time')
        if time_tag:
            post_date = parse_date(time_tag.get('datetime'))
            
            if post_date and START_DATE <= post_date <= END_DATE:
                # Select the author and the text
                author = post.find('p', class_='author')
                author_name = author.find('strong').get_text() if author and author.find('strong') else "Unknown"
                content = post.find('div', class_='content').get_text(strip=True)
                
                posts_data.append({
                    'date': post_date,
                    'author': author_name,
                    'text': content,
                    'url': thread_url
                })
    return posts_data

def scrape_bogleheads_range(pages=2):
    results = []
    
    for p in range(pages):
        start_val = p * 50
        url = f"{FORUM_URL}&start={start_val}"
        # print(f"Scraping forum index page {p+1}...")
        
        response = session.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Threads are usually in 'li' elements with class 'row'
        threads = soup.select('li.row')
        
        for thread in threads:
            # Check the last post time to see if we should skip this thread
            last_post_time = thread.find('time')
            if last_post_time:
                last_active = parse_date(last_post_time.get('datetime'))
                
                # Logic: If the thread was active after our START_DATE, it might contain relevant posts
                if last_active and last_active >= START_DATE:
                    link_tag = thread.find('a', class_='topictitle')
                    if link_tag:
                        # Build absolute URL
                        relative_link = link_tag['href'].lstrip('./')
                        thread_link = urllib.parse.urljoin(BASE_URL, relative_link)
                        
                        print(f"  Checking thread: {link_tag.text[:50]}")
                        thread_data = get_thread_posts(thread_link)
                        results.extend(thread_data)
                        
                        time.sleep(1) # Be respectful to avoid IP bans
                
                # If the thread hasn't been active since before our window, we've gone back far enough
                if last_active and last_active < START_DATE:
                    print("Reached end of date range in index.")
                    return pd.DataFrame(results)

    return pd.DataFrame(results)

In [ ]:
### https://www.bogleheads.org/forum/viewforum.php?f=2&start=[27000-33000] is what we are interested in!
df = scrape_bogleheads_range(pages=6000)
print(f"Found {len(df)} posts.")

Found 0 posts.


In [13]:
df.shape

(0, 0)

In [ ]:
df.to_csv('bogleheads_finance_2021_2022.csv', index=False)

In [ ]:
## google AI studio: DID NOT WORK

import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time

# Configuration
BASE_URL = "https://www.bogleheads.org/forum/viewforum.php"
FORUM_ID = 2
START_OFFSET = 27000  # Estimated offset for mid-2022
TOPICS_PER_PAGE = 50
START_DATE = datetime(2021, 11, 23)
END_DATE = datetime(2022, 6, 25)

def parse_date(date_str):
    # Bogleheads format example: 'Mon Nov 29, 2021 1:45 pm'
    # Format may vary slightly, simplified handler:
    try:
        return datetime.strptime(date_str.strip(), '%a %b %d, %Y %I:%M %p')
    except ValueError:
        return None

data = []
current_start = START_OFFSET
keep_scraping = True

print(f"Starting scrape from offset {current_start}...")

while keep_scraping:
    params = {'f': FORUM_ID, 'start': current_start}
    response = requests.get(BASE_URL, params=params)
    if response.status_code != 200:
        print("Failed to retrieve page")
        break
    
    soup = BeautifulSoup(response.text, 'html.parser')
    # phpBB topic rows usually have class 'row bg1' or 'row bg2'
    topic_rows = soup.select('li.row dl.icon')
    
    if not topic_rows:
        print("No topics found, stopping.")
        break
        
    page_topics_processed = 0
    
    for row in topic_rows:
        # Check if it is a sticky topic (usually separated or marked)
        # Using crude check for sticky icon or section separation if needed
        # For simplicity, we filter strictly by date
        
        # Extract Last Post Date
        last_post_div = row.find('dd', class_='lastpost')
        if not last_post_div: 
            continue
            
        date_span = last_post_div.find('span')
        if not date_span:
            # Sometimes date is directly text in dd or inside <time>
            date_text = last_post_div.get_text().split('\n')[0]
        else:
            date_text = date_span.get_text().strip()
            
        # Clean extra text like 'by User \n date'
        # This extraction depends heavily on exact HTML structure
        # Assuming date_text contains the date string
        try:
            # Splitting to isolate date part if 'by User' is present
            raw_date = date_text.splitlines()[0] if date_text else ""
            # Remove 'Today' or 'Yesterday' logic if present or parse strict dates
            # This is a placeholder for the actual text extraction logic
            post_date = parse_date(raw_date)
        except:
            continue

        if not post_date:
            continue

        # Logic: We are moving backwards in time (increasing start offset)
        # If post_date is NEWER than END_DATE, we skip it (too recent)
        # If post_date is OLDER than START_DATE, we stop scraping entirely
        
        if post_date > END_DATE:
            continue
        elif post_date < START_DATE:
            print(f"Reached date {post_date}, which is older than start date. Stopping.")
            keep_scraping = False
            break
        else:
            # Inside target window
            title = row.find('a', class_='topictitle').text.strip()
            link = row.find('a', class_='topictitle')['href']
            data.append({'date': post_date, 'title': title, 'link': link})
            page_topics_processed += 1

    print(f"Offset {current_start}: Collected {page_topics_processed} topics.")
    
    # Move to next page (older posts)
    current_start += TOPICS_PER_PAGE
    time.sleep(1) # Be polite

df = pd.DataFrame(data)
print(f"Scrape complete. Total rows: {len(df)}")

Starting scrape from offset 27000...
Failed to retrieve page
Scrape complete. Total rows: 0


In [3]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
from datetime import datetime
import urllib.parse

# Configuration
# Start at 27000 to get back to approx late 2022/early 2021
START_INDEX = 27000 
FORUM_BASE = "https://www.bogleheads.org/forum/viewforum.php?f=2"
BASE_URL = "https://www.bogleheads.org/forum/"
START_DATE = datetime(2021, 11, 23)
END_DATE = datetime(2022, 6, 25)

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36',
})

def parse_date(date_str):
    if not date_str: return None
    try:
        # Handling the 'Z' or offset in ISO format
        return datetime.fromisoformat(date_str.replace('Z', '+00:00').split('+')[0])
    except:
        return None

def get_thread_posts(thread_url):
    posts_data = []
    try:
        response = session.get(thread_url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        posts = soup.find_all('div', class_='post')
        
        for post in posts:
            time_tag = post.find('time')
            if time_tag:
                post_date = parse_date(time_tag.get('datetime'))
                if post_date and START_DATE <= post_date <= END_DATE:
                    author_tag = post.find('p', class_='author')
                    author_name = author_tag.find('strong').get_text() if author_tag and author_tag.find('strong') else "Unknown"
                    content = post.find('div', class_='content').get_text(strip=True)
                    
                    posts_data.append({
                        'date': post_date, 
                        'author': author_name, 
                        'text': content, 
                        'url': thread_url
                    })
    except Exception as e:
        print(f"Error reading thread {thread_url}: {e}")
    return posts_data

def scrape_bogleheads_range(num_pages=10):
    results = []
    
    for p in range(num_pages):
        # Correctly calculate the offset starting from 27000
        current_start = START_INDEX + (p * 50)
        url = f"{FORUM_BASE}&start={current_start}"
        
        print(f"Accessing Index: {current_start}...")
        response = session.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')
        threads = soup.select('li.row')
        
        if not threads:
            print("No more threads found.")
            break

        for thread in threads:
            # Check the date of the thread's last post
            last_post_time = thread.find('time')
            if last_post_time:
                last_active = parse_date(last_post_time.get('datetime'))
                
                # Logic: If thread was active after our window started, it's worth checking
                if last_active and last_active >= START_DATE:
                    link_tag = thread.find('a', class_='topictitle')
                    if link_tag:
                        thread_link = urllib.parse.urljoin(BASE_URL, link_tag['href'].lstrip('./'))
                        # Only enter if the thread's last activity is within or after our range
                        thread_data = get_thread_posts(thread_link)
                        results.extend(thread_data)
                        time.sleep(0.5) # Slight delay
                
                # Logic: If we see threads that haven't been touched since before Nov 2021, we stop
                if last_active and last_active < START_DATE:
                    print(f"Reached old content ({last_active}). Stopping.")
                    return pd.DataFrame(results)

    return pd.DataFrame(results)

# Run with a smaller page count first to verify
df = scrape_bogleheads_range(num_pages=10000) 
print(f"Successfully found {len(df)} posts.")
if not df.empty:
    df.to_csv('bogleheads_finance_range.csv', index=False)

Accessing Index: 27000...
No more threads found.
Successfully found 0 posts.


In [4]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
import urllib.parse
from datetime import datetime

# --- CONFIGURATION ---
FORUM_URL = "https://www.bogleheads.org/forum/viewforum.php?f=2"
BASE_URL = "https://www.bogleheads.org/forum/"
START_TOPIC = 27000
END_TOPIC = 33000
STEP = 50
OUTPUT_FILE = f"bogleheads_topics_{START_TOPIC}_{END_TOPIC}.csv"

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
})

def get_thread_content(thread_url):
    """Scrapes all posts/comments on the first page of a specific thread."""
    posts = []
    try:
        response = session.get(thread_url, timeout=10)
        if response.status_code != 200:
            return []
            
        soup = BeautifulSoup(response.text, 'html.parser')
        post_elements = soup.find_all('div', class_='post')
        
        for p in post_elements:
            # Extract basic post data
            author_tag = p.find('p', class_='author')
            author = author_tag.find('strong').text if author_tag and author_tag.find('strong') else "Unknown"
            
            content_tag = p.find('div', class_='content')
            content = content_tag.get_text(strip=True) if content_tag else ""
            
            time_tag = p.find('time')
            timestamp = time_tag['datetime'] if time_tag else "N/A"
            
            posts.append({
                'author': author,
                'text': content,
                'date': timestamp,
                'url': thread_url
            })
    except Exception as e:
        print(f"  [!] Error scraping {thread_url}: {e}")
    return posts

def run_scrape():
    all_data = []
    start_time = datetime.now()
    
    print(f"--- Starting Scrape at {start_time.strftime('%H:%M:%S')} ---")
    
    for current_start in range(START_TOPIC, END_TOPIC + STEP, STEP):
        print(f"Index: {current_start} / {END_TOPIC}...")
        
        try:
            url = f"{FORUM_URL}&start={current_start}"
            response = session.get(url)
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Select all links that lead to threads
            topics = soup.find_all('a', class_='topictitle')
            
            if not topics:
                print(f"No more threads found at offset {current_start}.")
                break
                
            for topic in topics:
                # Build the full URL
                relative_link = topic['href'].lstrip('./')
                thread_link = urllib.parse.urljoin(BASE_URL, relative_link)
                
                # Fetch posts from the thread
                thread_posts = get_thread_content(thread_link)
                all_data.extend(thread_posts)
                
                # Be respectful to the server
                time.sleep(0.8)
                
        except Exception as e:
            print(f"Error on index offset {current_start}: {e}")
            
    # Save to CSV
    df = pd.DataFrame(all_data)
    df.to_csv(OUTPUT_FILE, index=False)
    
    end_time = datetime.now()
    duration = end_time - start_time
    print(f"--- Finished! ---")
    print(f"Total Posts Scraped: {len(df)}")
    print(f"Time Taken: {duration}")
    print(f"File Saved: {OUTPUT_FILE}")
    
    return df

##To execute the script, uncomment the line below:
df = run_scrape()

--- Starting Scrape at 13:35:53 ---
Index: 27000 / 33000...
No more threads found at offset 27000.
--- Finished! ---
Total Posts Scraped: 0
Time Taken: 0:00:00.074979
File Saved: bogleheads_topics_27000_33000.csv


In [9]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
import urllib.parse

# --- CONFIGURATION ---
FORUM_URL = "https://www.bogleheads.org/forum/viewforum.php?f=2"
BASE_URL = "https://www.bogleheads.org/forum/"
START_TOPIC = 27000
END_TOPIC = 33000
STEP = 50 

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
})

def scrape_thread_pages(thread_url, topic_title):
    """Navigates through all pages of a thread and scrapes every post."""
    all_thread_posts = []
    current_url = thread_url
    
    while current_url:
        try:
            response = session.get(current_url, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')
            posts = soup.select('div.post')
            
            for i, post in enumerate(posts):
                # Distinguish between Original Post and Reply
                is_first_post = (i == 0 and "&start=" not in current_url)
                
                user = post.select_one('p.author strong')
                content = post.select_one('div.content')
                timestamp = post.select_one('time')
                
                all_thread_posts.append({
                    'topic_title': topic_title,
                    'type': 'Original Post' if is_first_post else 'Reply',
                    'username': user.get_text() if user else "Unknown",
                    'content': content.get_text(strip=True) if content else "",
                    'time': timestamp['datetime'] if timestamp else "N/A",
                    'url': current_url
                })
            
            # Check for "Next" page in the thread
            next_page = soup.select_one('li.arrow.next a')
            if next_page:
                current_url = urllib.parse.urljoin(BASE_URL, next_page['href'].lstrip('./'))
                time.sleep(1) # Respectful delay
            else:
                current_url = None # No more pages in this thread
                
        except Exception as e:
            print(f"  [!] Error on child page {current_url}: {e}")
            break
            
    return all_thread_posts

def run_master_scrape():
    final_dataset = []
    
    for offset in range(START_TOPIC, END_TOPIC + STEP, STEP):
        index_url = f"{FORUM_URL}&start={offset}"
        print(f"Scraping Index Offset: {offset}...")
        
        try:
            response = session.get(index_url)
            print(index_url)
            soup = BeautifulSoup(response.text, 'html.parser')
            topic_links = soup.find_all('a', class_='topictitle')
            
            if not topic_links:
                print(f"No topics found at offset {offset}. Ending scrape.")
                break

            for link in topic_links:
                title = link.get_text()
                child_url = urllib.parse.urljoin(BASE_URL, link['href'].lstrip('./'))
                
                print(f"  Deep-scraping topic: {title[:50]}...")
                thread_data = scrape_thread_pages(child_url, title)
                final_dataset.extend(thread_data)
                
                # Save progress incrementally to avoid data loss
                if len(final_dataset) % 500 == 0:
                    pd.DataFrame(final_dataset).to_csv('bogleheads_partial.csv', index=False)
                
                time.sleep(1) 
                
        except Exception as e:
            print(f"Error at index {offset}: {e}")

    df = pd.DataFrame(final_dataset)
    df.to_csv('bogleheads_final_dataset.csv', index=False)
    print(f"Done! Total rows captured: {len(df)}")

run_master_scrape()

Scraping Index Offset: 27000...
https://www.bogleheads.org/forum/viewforum.php?f=2&start=27000
No topics found at offset 27000. Ending scrape.
Done! Total rows captured: 0


In [10]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
import urllib.parse
import os

# --- CONFIGURATION ---
# IMPORTANT: Adjust START_TOPIC if you get "No topics found". 
# 27000 may be empty; try 15000 if you see no results.
FORUM_URL = "https://www.bogleheads.org/forum/viewforum.php?f=2"
BASE_URL = "https://www.bogleheads.org/forum/"
START_TOPIC = 27000 
END_TOPIC = 33000
STEP = 50 

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.5'
})

def scrape_thread_pages(thread_url, topic_title):
    all_posts = []
    current_url = thread_url
    
    while current_url:
        try:
            res = session.get(current_url, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')
            
            # Find all post containers
            post_containers = soup.select('div.post')
            
            for i, p in enumerate(post_containers):
                # Metadata extraction
                user = p.select_one('p.author strong')
                content = p.select_one('div.content')
                timestamp = p.select_one('time')
                
                all_posts.append({
                    'topic_title': topic_title,
                    'row_type': 'Original Post' if (i == 0 and "start=" not in current_url) else 'Reply',
                    'username': user.get_text(strip=True) if user else "Unknown",
                    'content': content.get_text(strip=True) if content else "",
                    'time': timestamp['datetime'] if timestamp else "N/A",
                    'url': current_url
                })
            
            # Check for "Next" page inside the thread
            next_btn = soup.select_one('li.arrow.next a')
            if next_btn:
                current_url = urllib.parse.urljoin(BASE_URL, next_btn['href'].lstrip('./'))
                time.sleep(1)
            else:
                current_url = None
        except Exception as e:
            print(f"  [!] Error in child page: {e}")
            break
    return all_posts

def run_master_scrape():
    final_data = []
    
    for offset in range(START_TOPIC, END_TOPIC + STEP, STEP):
        url = f"{FORUM_URL}&start={offset}"
        print(f"Scraping Index Offset: {offset}...")
        
        res = session.get(url)
        soup = BeautifulSoup(res.text, 'html.parser')
        
        # STRUCTURE CHECK: Try multiple ways to find topictitle
        # 1. Standard class 2. Nested in dt 3. Any viewtopic link
        topic_links = soup.select('a.topictitle') or soup.select('dt a.topictitle')
        
        if not topic_links:
            # DEBUG: Save HTML to see why it failed
            filename = f"debug_offset_{offset}.html"
            with open(filename, "w", encoding="utf-8") as f:
                f.write(res.text)
            print(f"  [!] No topics found. HTML saved to {filename}. Check if page is empty or blocked.")
            continue

        for link in topic_links:
            title = link.get_text(strip=True)
            child_url = urllib.parse.urljoin(BASE_URL, link['href'].lstrip('./'))
            
            print(f"    Entering Topic: {title[:50]}...")
            thread_rows = scrape_thread_pages(child_url, title)
            final_data.extend(thread_rows)
            time.sleep(1) # Safety delay
            
        # Optional: Save every index page to avoid data loss
        pd.DataFrame(final_data).to_csv('bogleheads_incremental.csv', index=False)

    df = pd.DataFrame(final_data)
    df.to_csv('bogleheads_final.csv', index=False)
    print(f"Scrape Finished. Total Rows: {len(df)}")

run_master_scrape()

Scraping Index Offset: 27000...
  [!] No topics found. HTML saved to debug_offset_27000.html. Check if page is empty or blocked.
Scraping Index Offset: 27050...
  [!] No topics found. HTML saved to debug_offset_27050.html. Check if page is empty or blocked.
Scraping Index Offset: 27100...
  [!] No topics found. HTML saved to debug_offset_27100.html. Check if page is empty or blocked.
Scraping Index Offset: 27150...
  [!] No topics found. HTML saved to debug_offset_27150.html. Check if page is empty or blocked.
Scraping Index Offset: 27200...
  [!] No topics found. HTML saved to debug_offset_27200.html. Check if page is empty or blocked.
Scraping Index Offset: 27250...
  [!] No topics found. HTML saved to debug_offset_27250.html. Check if page is empty or blocked.
Scraping Index Offset: 27300...
  [!] No topics found. HTML saved to debug_offset_27300.html. Check if page is empty or blocked.
Scraping Index Offset: 27350...
  [!] No topics found. HTML saved to debug_offset_27350.html. Che

In [14]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import random

# --- CONFIGURATION ---
START_TOPIC = 30000
END_TOPIC = 33000
FORUM_URL = f"https://www.bogleheads.org/forum/viewforum.php?f=2&start={START_TOPIC}"

def setup_driver():
    chrome_options = Options()
    # Adding arguments to make the browser look less like a bot
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    # chrome_options.add_argument("--headless") # Uncomment to run without a window opening
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    # Overwrite the 'webdriver' flag so websites can't detect it easily
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver

def scrape_thread_with_selenium(driver, thread_url, topic_title):
    thread_data = []
    driver.get(thread_url)
    
    while True:
        # Wait for posts to load
        try:
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "post")))
            posts = driver.find_elements(By.CLASS_NAME, "post")
            
            for i, p in enumerate(posts):
                try:
                    user = p.find_element(By.CSS_SELECTOR, "p.author strong").text
                    content = p.find_element(By.CLASS_NAME, "content").text
                    timestamp = p.find_element(By.TAG_NAME, "time").get_attribute("datetime")
                    
                    thread_data.append({
                        'topic_title': topic_title,
                        'row_type': 'Original Post' if i == 0 and "start=" not in driver.current_url else 'Reply',
                        'username': user,
                        'content': content,
                        'time': timestamp,
                        'url': driver.current_url
                    })
                except:
                    continue
            
            # Check for pagination (Next button) within the thread
            next_buttons = driver.find_elements(By.CSS_SELECTOR, "li.arrow.next a")
            if next_buttons:
                next_buttons[0].click()
                time.sleep(random.uniform(2, 4))
            else:
                break # No more pages in this thread
        except:
            break
            
    return thread_data

def run_master_scrape():
    driver = setup_driver()
    final_data = []
    
    try:
        # 1. Navigate to the Index Page
        driver.get(FORUM_URL)
        time.sleep(random.uniform(5, 7)) # Wait for Cloudflare to clear
        
        # 2. Collect all topic links on the current index page
        # Note: Bogleheads uses 'topictitle' class
        topic_elements = driver.find_elements(By.CLASS_NAME, "topictitle")
        topics_to_visit = []
        for te in topic_elements:
            topics_to_visit.append({
                'title': te.text,
                'url': te.get_attribute("href")
            })

        print(f"Found {len(topics_to_visit)} topics. Starting deep scrape...")

        # 3. Visit each "Child" page
        for topic in topics_to_visit:
            print(f"  Scraping: {topic['title'][:50]}")
            data = scrape_thread_with_selenium(driver, topic['url'], topic['title'])
            final_data.extend(data)
            
            # Anti-bot delay
            time.sleep(random.uniform(3, 6))
            
            # Intermediate save
            pd.DataFrame(final_data).to_csv("bogleheads_selenium_30_33k.csv", index=False)

    finally:
        driver.quit()
        
    df = pd.DataFrame(final_data)
    df.to_csv("bogleheads_selenium_4.csv", index=False)
    print(f"Done! Captured {len(df)} rows.")

run_master_scrape()

Found 51 topics. Starting deep scrape...
  Scraping: Personal Finance Forum Posting Guidelines
  Scraping: Buying too much house?
  Scraping: Annual Bloodwork?
  Scraping: AMT Foreign Tax Credit: pros and cons of using the
  Scraping: Nice plug for Open Social Security
  Scraping: What should I do next? Done maxing accounts for 20
  Scraping: How to Handle Inherited Accounts at Fidelity?
  Scraping: QBI- which SE deductible amount?
  Scraping: To buy a house or not? That is the question
  Scraping: Mega Backdoor conversion taxation and 1099 R query
  Scraping: Is 1116 Needed?
  Scraping: Roth IRA rollover schedule advice
  Scraping: Trust Question - Family member passed
  Scraping: tax filing question
  Scraping: real-estate: forgot deductions for depreciation fo
  Scraping: PayPal 1099–k for online gambling
  Scraping: Managing Parent's Finances
  Scraping: Possible switch to employer offering 403b and 457
  Scraping: I made a mess with 1099-Q
  Scraping: Mortgage: 30 yr, 15 yr, or ca